# SoccerNet GSR — Colab Inference

Runs the SoccerNet Game State Reconstruction baseline on Colab with GPU.

**Setup:** Runtime → Change runtime type → T4 GPU

**Note:** Jersey number detection (MMOCR) is skipped — it requires mmcv which has no pre-built wheels for Colab's Python 3.12 + torch 2.x. All other pipeline stages work.

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

## Setup — Install dependencies

This cell:
1. Clones our repo + sn-gamestate
2. Patches sn-gamestate's version pins for Colab compatibility
3. Installs dependencies (skips mmcv/mmocr — jersey number detection not available on Colab)

In [ ]:
import os

# === Step 1: Clone repos ===
REPO_URL = "https://github.com/Moiz005/SoccerVision-Player-Tracking-3D-Reconstruction.git"
REPO_NAME = "SoccerVision-Player-Tracking-3D-Reconstruction"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

if not os.path.exists("sn-gamestate"):
    !git clone https://github.com/SoccerNet/sn-gamestate.git

# === Step 2: Patch sn-gamestate's pyproject.toml ===
# Relax version pins that block Colab (Python 3.12, torch 2.x)
pyproject_path = "sn-gamestate/pyproject.toml"
with open(pyproject_path, "r") as f:
    content = f.read()

# Allow Python 3.10+
content = content.replace(
    'requires-python = ">=3.9,<3.10"',
    'requires-python = ">=3.9"'
)

# Remove torch version pin
content = content.replace(
    '    "torch==1.13.1",',
    '    "torch",'
)

# Float numpy for Python 3.12 compat
content = content.replace(
    '    "numpy==1.26.4",',
    '    "numpy>=1.26.4",'
)

with open(pyproject_path, "w") as f:
    f.write(content)

print("Patched pyproject.toml: relaxed requires-python, torch pin, numpy pin")

# === Step 3: Install sn-gamestate without resolving deps ===
%cd sn-gamestate
!pip install --no-deps -e .

# === Step 4: Install remaining deps manually ===
# Skips: mmcv, mmdet, mmocr (no pre-built wheels for Colab Python 3.12 + torch 2.x)
!pip install "tracklab==1.3.24" \
    "soccernet==0.1.55" \
    "lightning==2.0.9" \
    "transformers==4.35.2" \
    "tokenizers==0.15.2" \
    "easyocr==1.7.1"

print("\n=== All dependencies installed (jersey number detection skipped) ===")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

import tracklab
print(f"TrackLab: {tracklab.__version__}")

import sn_gamestate
print("sn-gamestate: OK")

print("\nCore dependencies verified! (mmcv/mmocr not installed — jersey detection unavailable)")

## Run Baseline on Official Validation Video

This runs the GSR pipeline on **1 validation video** from SoccerNet:
- Player detection (YOLOv11)
- Re-ID & tracking (PRTReid + StrongSORT)
- Pitch localization (NBW calibration)
- Team assignment (K-means)

Jersey number detection is **skipped** (requires mmcv).  
First run auto-downloads ~35GB of dataset + model weights.

In [ ]:
import os
os.chdir(f"/content/{REPO_NAME}/sn-gamestate")

# Run pipeline WITHOUT jersey_number_detect (mmcv not available)
!tracklab -cn soccernet \
    ~modules.jersey_number_detect \
    pipeline='[bbox_detector,reid,track,pitch,calibration,tracklet_agg,team,team_side]'

## View Results

The pipeline generates:
- Annotated visualization video (bounding boxes, IDs, pitch overlay)
- Tracker state file (.pklz)
- Evaluation metrics (GS-HOTA)

In [ ]:
import glob
from IPython.display import Video, display

output_videos = glob.glob(
    "output/**/visualization/videos/*.mp4",
    recursive=True
)

print(f"Found {len(output_videos)} output video(s):")
for v in output_videos:
    print(f"  - {v}")

if output_videos:
    print(f"\nPlaying: {output_videos[0]}")
    display(Video(output_videos[0], width=800))
else:
    print("No output videos found. Check the pipeline output above for errors.")

## Next Steps

If this ran successfully, the baseline works on official data.

**Next:** Run this on your own football clip → Phase 4 (Custom Video Adapter)